In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import seaborn as sns
import numpy as np
import random
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
def get_file(path):
    """
    recupère le fichier csv donnée avec le séparateur ';' et retourne un dataFrame.
    
    Args:
        path (str): Le chemin vers le fichier CSV.
        
    Returns:
        pd.DataFrame: Les données chargées.
    """
    
    #ouvir le fichier en dataframe
    df = pd.read_csv(path, sep=',')
    return df

In [3]:
def cal_fiabilite(valeurs_attendues, valeurs_predites, noms_colonnes):
    resultats = {}
    somme_taux = 0
    
    # On compare colonne par colonne
    for i, col_name in enumerate(noms_colonnes):
        vrais = valeurs_attendues[col_name]
        preds = valeurs_predites[:, i] 
        
        # Calcul de la fiabilité
        mae = mean_absolute_error(vrais, preds)
        moyenne_reelle = np.mean(vrais)
        
        if moyenne_reelle != 0:
            taux = max(0, (1 - (mae / moyenne_reelle)) * 100)
        else:
            taux = 0.0
            
        # On stocke le résultat de cette colonne
        resultats[col_name] = taux
        somme_taux += taux
        
    # On calcule et on stocke la moyenne globale
    resultats["Global"] = somme_taux / len(noms_colonnes)
    
    return resultats

In [124]:
df = get_file("cleaned_dataset.csv")

input_col = ['Annee',
             'Mois',
             'Service',
             "Gare de départ",
             "Gare d'arrivée",
             "Durée moyenne du trajet",
             "Nombre de circulations prévues"
            ]

output_col = [#"Nombre de trains annulés",
              "Nombre de trains en retard au départ",
              #"Retard moyen des trains en retard au départ",
              "Nombre de trains en retard à l'arrivée",
              "Retard moyen des trains en retard à l'arrivée",
              "Retard moyen de tous les trains à l'arrivée",
              "Nombre trains en retard > 15min",
              #"Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
              #"Nombre trains en retard > 30min",
              #"Nombre trains en retard > 60min",
              #"Prct retard pour causes externes",
              #"Prct retard pour cause infrastructure",
              #"Prct retard pour cause gestion trafic",
              #"Prct retard pour cause matériel roulant",
              #"Prct retard pour cause gestion en gare et réutilisation de matériel",
              #"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
             ]
LE = LabelEncoder()
for col in ['Service', "Gare de départ", "Gare d'arrivée"]:
    df[col] = LE.fit_transform(df[col])

print (df.shape)

inputt = df[input_col]
output = df[output_col]

input_train, input_test, output_train, output_test = train_test_split(inputt, output, test_size=0.1)


modele_rf = RandomForestRegressor(n_estimators=100, n_jobs=-1)
modele_rf.fit(input_train, output_train)
prediction_rf = modele_rf.predict(input_test)

model = DecisionTreeRegressor()
model.fit(input_train, output_train)
prediction = model.predict(input_test)

scores_rf = cal_fiabilite(output_test, prediction_rf, output_col)
scores = cal_fiabilite(output_test, prediction, output_col)


print(f"Score Forest : {scores_rf['Global']} ")
print(f"Score Tree : {scores['Global']} ")

df_predictions = pd.DataFrame(prediction_rf, columns=output_col)
df_predictions

(11837, 29)
Score Forest : 70.88264089827469 
Score Tree : 59.91569657036414 


,Nombre de trains en retard au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,Retard moyen de tous les trains à l'arrivée,Nombre trains en retard > 15min
0,41.65,12.98,45.5426,7.5928,10.65
1,20.22,31.12,30.9407,11.2073,29.92
2,22.16,37.65,44.6259,11.0691,37.23
3,44.18,19.16,32.4619,4.2347,17.14
4,152.96,17.25,24.0289,2.6010,9.93
...,...,...,...,...,...
1179,148.88,110.76,37.4176,5.1654,83.47
1180,191.14,80.81,28.5186,9.0830,60.65
1181,16.90,24.53,28.0978,4.8388,15.00
1182,70.86,53.93,24.0731,3.4203,29.31
